# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ssarimm/Machine-Learning-Internship---Flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice and why

I will use a **Decision Tree Classifier** to identify pages whose observed `trend_direction` is `down`. The target is a proxy for current decline, not a direct measure of whether a content refresh will succeed.

A Decision Tree fits this first modeling pass because it can learn non-linear patterns, handle mixed numeric and categorical features after preprocessing, and provide interpretable decision rules. I will compare its ranking with my Week 4 baseline, which assigns 2 points to stale pages and 1 point to pages with high search demand.

I will use Precision@50 as the primary decision metric: of the 50 highest-ranked pages, what fraction have the proxy label `down`? Both approaches must be evaluated on the same held-out clients.

I will exclude identifiers and the target-derived fields `trend_direction` and `trend_pct` from the model features. Results will be treated as observed, directional decision-support evidence. They will not prove that refreshing a page causes traffic to recover.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use an 80/20 holdout split grouped by `client_id`. This means all pages belonging to a client stay together in either the training set or the test set, rather than appearing in both.

The model will learn from the training clients and be evaluated on clients it has not seen. I will use a fixed random seed of 42 so the split is reproducible.

The target is the observed label `trend_direction == "down"`. Since this label describes the supplied dataset's existing trend classification rather than a future outcome, this experiment evaluates classification of the proxy label; it does not establish future forecasting skill.

The Week 4 baseline and Decision Tree will be evaluated on exactly the same test clients using Precision@50 and Precision@100.

In [7]:
import os

print("Current directory:", os.getcwd())
print("\nFiles and folders here:")
print(os.listdir())

Current directory: /content

Files and folders here:
['.config', 'sample_data']


In [8]:
import os

filename = "content_refresh_anonymized.csv"
search_roots = ["/content", "/content/drive/MyDrive"]

matches = []

for root in search_roots:
    if os.path.exists(root):
        for current_dir, dirs, files in os.walk(root):
            # Skip large hidden/system folders
            dirs[:] = [
                d for d in dirs
                if not d.startswith(".") and d != "node_modules"
            ]
            if filename in files:
                matches.append(os.path.join(current_dir, filename))

print("Matches found:")
for path in matches:
    print(path)

Matches found:


In [10]:
import pandas as pd

df = pd.read_csv(r"/content/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (30000, 53)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_pct,health_score,needs_indexing,is_quick_win,needs_ctr_fix,needs_engagement_fix,ai_opportunity,is_underperformer,is_declining,is_initial_refresh_candidate
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,-41.4,50,False,False,False,False,False,False,True,True
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,-57.7,40,False,True,False,False,False,False,True,True
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,-60.9,40,False,False,False,False,False,False,True,False
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,-13.8,60,False,False,False,True,False,False,False,True
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,-34.7,40,False,False,False,True,True,False,True,False


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit

# Numeric features
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

# Categorical features
categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
]

feature_columns = numeric_features + categorical_features

# Check that every selected feature exists
missing = set(feature_columns + [
    "client_id", "content_id", "trend_direction"
]) - set(df.columns)

assert not missing, f"Missing required columns: {missing}"

# Keep only rows with a known target and client group
model_df = df.dropna(
    subset=["client_id", "trend_direction"]
).copy()

# Target: 1 means the observed trend label is "down"
y = (model_df["trend_direction"] == "down").astype(int)

# Model input: do not include IDs or target-derived columns
X = model_df[feature_columns].copy()

# Hold out entire clients, not random individual pages
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=model_df["client_id"])
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(model_df.iloc[train_idx]["client_id"])
test_clients = set(model_df.iloc[test_idx]["client_id"])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Clients shared between sets:", len(train_clients & test_clients))
print("Training decline rate:", round(y_train.mean(), 4))
print("Test decline rate:", round(y_test.mean(), 4))

assert len(train_clients & test_clients) == 0
assert len(X_train) > 0 and len(X_test) > 0

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Clients shared between sets: 0
Training decline rate: 0.5501
Test decline rate: 0.511


In [12]:
print("trend_direction exists:", "trend_direction" in df.columns)
print("is_declining exists:", "is_declining" in df.columns)

if "trend_direction" in df.columns:
    print(df["trend_direction"].value_counts(dropna=False))

trend_direction exists: True
is_declining exists: True
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Train and compare with the Week 4 baseline

I will train a Decision Tree Classifier using the training clients and evaluate it on the held-out test clients. Numeric missing values will be imputed with the training-set median, and categorical features will be one-hot encoded.

I will compare the model against the Week 4 rule-based baseline, which gives 2 points to stale pages (at least 180 days since their last update) and 1 point to pages with high search demand. The high-demand threshold will be calculated from the training data only.

Both methods will rank the same test pages. Precision@50 and Precision@100 will measure how many of the top-ranked pages have the observed decline label. These results evaluate agreement with the existing decline label, not whether refreshing a page will recover its traffic.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# 1. Preprocess numeric and categorical features
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# 2. Train the Decision Tree
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=30,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

# 3. Get the model's decline probabilities for test pages
model_scores = model.predict_proba(X_test)[:, 1]

# 4. Recreate the Week 4 baseline on the same test pages
test_frame = model_df.iloc[test_idx].copy()

positive_train_volume = X_train.loc[
    X_train["search_volume"] > 0, "search_volume"
].dropna()

high_demand_threshold = positive_train_volume.median()

baseline_stale = (
    test_frame["days_since_last_update"] >= 180
).fillna(False)

baseline_high_demand = (
    test_frame["search_volume"].notna()
    & (test_frame["search_volume"] > 0)
    & (test_frame["search_volume"] >= high_demand_threshold)
)

baseline_scores = (
    baseline_stale.astype(int) * 2
    + baseline_high_demand.astype(int)
)

# 5. Precision at K: fraction of top-ranked pages labeled declining
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    top_indices = np.argsort(-scores, kind="stable")[:k]
    return y_true[top_indices].mean()

# 6. Compare both methods
results = []

for method, scores in [
    ("Week 4 baseline", baseline_scores),
    ("Decision Tree", model_scores)
]:
    results.append({
        "method": method,
        "Precision@50": precision_at_k(y_test, scores, 50),
        "Precision@100": precision_at_k(y_test, scores, 100),
        "ROC-AUC": roc_auc_score(y_test, scores),
        "Average Precision": average_precision_score(y_test, scores)
    })

results_df = pd.DataFrame(results)
display(results_df.round(3))

print("High-demand threshold:", high_demand_threshold)
print("Test rows evaluated:", len(y_test))


,method,Precision@50,Precision@100,ROC-AUC,Average Precision
0,Week 4 baseline,0.66,0.59,0.486,0.504
1,Decision Tree,0.42,0.41,0.608,0.580


High-demand threshold: 20.0
Test rows evaluated: 6163


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Error analysis and interpretation

At a classification threshold of 0.5, the Decision Tree produced 1,445 false negatives and 1,103 false positives on the held-out test set.

False negatives are pages labeled as declining by the dataset that the model did not flag. One example had 92 days since its last update and a search volume of 20, but received a predicted decline probability of approximately 0.006. This suggests that the model's learned feature patterns do not capture every page labeled as declining.

False positives are pages the model flagged as declining even though their observed trend label was not `down`. One example was labeled `stable` but received a predicted probability of approximately 0.970. Such cases may be worth reviewing to understand which feature combinations lead to high predictions.

Some pages receive identical predicted probabilities, including pages from the same client. This may reflect pages falling into the same decision-tree leaf. The error counts depend on the chosen 0.5 threshold, so they should be interpreted alongside the ranking metrics.

The Week 4 baseline achieved better Precision@50 and Precision@100, while the Decision Tree achieved better ROC-AUC and Average Precision. Therefore, the baseline remains stronger for the immediate top-ranked review queue in this experiment, while the Decision Tree offers a different overall ranking. Neither method establishes that refreshing a page will recover its traffic.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Align predictions and labels with the test rows
error_review = test_frame[
    ["content_id", "client_id", "trend_direction",
     "days_since_last_update", "search_volume"]
].copy()

error_review["actual_decline"] = np.asarray(y_test)
error_review["model_score"] = model_scores
error_review["baseline_score"] = np.asarray(baseline_scores)

# Decision Tree classification at a 0.5 probability threshold
error_review["model_prediction"] = (model_scores >= 0.5).astype(int)

# False negatives: declining pages the model did not flag
false_negatives = error_review[
    (error_review["actual_decline"] == 1)
    & (error_review["model_prediction"] == 0)
].sort_values("model_score")

# False positives: pages flagged by the model but not labeled declining
false_positives = error_review[
    (error_review["actual_decline"] == 0)
    & (error_review["model_prediction"] == 1)
].sort_values("model_score", ascending=False)

print("False negatives:", len(false_negatives))
display(false_negatives.head(10))

print("False positives:", len(false_positives))
display(false_positives.head(10))

False negatives: 1445


,content_id,client_id,trend_direction,days_since_last_update,search_volume,actual_decline,model_score,baseline_score,model_prediction
27271,content_7bc32bc1df59,client_8527a891e2,down,92,20.0,1,0.006349,1,0
29224,content_bf212b1edf1a,client_434c9b5ae5,down,89,10.0,1,0.050826,0,0
8347,content_b003870be70a,client_434c9b5ae5,down,8,10.0,1,0.050826,0,0
25616,content_f8e170580eab,client_434c9b5ae5,down,8,0.0,1,0.050826,0,0
17117,content_cd050ee10f95,client_434c9b5ae5,down,8,10.0,1,0.050826,0,0
22346,content_14b91a8ba3cc,client_434c9b5ae5,down,8,40.0,1,0.050826,1,0
20831,content_420f30e62d90,client_8527a891e2,down,103,30.0,1,0.050826,1,0
14927,content_08366e95bd39,client_8527a891e2,down,92,0.0,1,0.050826,0,0
4358,content_94d39841847a,client_8527a891e2,down,104,40.0,1,0.050826,1,0
6140,content_f562786a3f25,client_434c9b5ae5,down,89,10.0,1,0.050826,0,0


False positives: 1103


,content_id,client_id,trend_direction,days_since_last_update,search_volume,actual_decline,model_score,baseline_score,model_prediction
6903,content_c84a0ab98e90,client_f369cb89fc,stable,20,0.0,0,0.970096,0,1
27589,content_0d61e38b3645,client_4e07408562,stable,13,1300.0,0,0.851294,1,1
4638,content_6bf79d7d0a4f,client_4e07408562,stable,13,20.0,0,0.851294,1,1
17859,content_5c5fab9d41e7,client_4e07408562,up,104,22200.0,0,0.851294,1,1
96,content_600405887c39,client_4e07408562,stable,13,260.0,0,0.851294,1,1
9097,content_32ff84795595,client_4e07408562,up,104,170.0,0,0.851294,1,1
14229,content_ccc3299baa8a,client_4e07408562,stable,13,880.0,0,0.851294,1,1
3163,content_eb7015b56be9,client_4e07408562,stable,13,2900.0,0,0.851294,1,1
19909,content_c83476e39f7d,client_4e07408562,stable,13,20.0,0,0.851294,1,1
17682,content_6b296ae2c02f,client_4e07408562,stable,13,2400.0,0,0.851294,1,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.